# 05 · Baselines for arrival delay before departure

This notebook establishes the minimum performance that later models must beat for
`arrival_pre`. It uses exactly the temporal Parquet splits produced by notebook 04.

Three leakage-safe baselines are compared:

1. the median arrival delay in train for every flight;
2. the train median for `ADEP → ADES`, falling back to the train median for `ADEP`
   and finally to the global train median;
3. the train median for `ADEP → ADES + AC Operator`, with a validation-selected
   minimum frequency and hierarchical fallback.

The realised departure delay is deliberately not used: it is unknown before departure.
That operational baseline belongs to a separate post-off-block task.


In [1]:
from pathlib import Path
import sys

from pyspark import StorageLevel
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spark_flight_pipeline import create_spark

DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "model" / "arrival_pre"
REPORT_PATH = PROJECT_ROOT / "reports" / "modeling" / "legacy" / "baseline" / "05_baseline_metrics.csv"
THRESHOLD_REPORT_PATH = REPORT_PATH.parent / "05_route_airline_threshold_search.csv"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
TARGET = "Arrival_Delay_Min"
MIN_ROUTE_OBSERVATIONS = 100
EXPECTED_COUNTS = {
    "train": 2_457_169,
    "validation": 591_391,
    "test": 622_698,
}
FORBIDDEN_COLUMNS = {
    "ACTUAL OFF BLOCK TIME",
    "ACTUAL ARRIVAL TIME",
    "Actual Distance Flown (nm)",
    "Departure_Delay_Min",
}


## Load the frozen temporal splits and verify their contract

Train is the only source used to estimate medians. Validation selects the baseline;
test is evaluated only after that choice is frozen.


In [2]:
spark = create_spark("arrival-pre-baselines", master="local[2]")
spark.conf.set("spark.sql.shuffle.partitions", "16")
splits = {
    name: spark.read.parquet(str(DATA_ROOT / name))
    for name in ("train", "validation", "test")
}
counts = {name: frame.count() for name, frame in splits.items()}
assert counts == EXPECTED_COUNTS, (counts, EXPECTED_COUNTS)
assert all(TARGET in frame.columns for frame in splits.values())
assert all(not (FORBIDDEN_COLUMNS & set(frame.columns)) for frame in splits.values())
assert len({frame.schema.json() for frame in splits.values()}) == 1
print({"counts": counts, "same_schema": True, "leakage_columns": []})


{'counts': {'train': 2457169, 'validation': 591391, 'test': 622698}, 'same_schema': True, 'leakage_columns': []}


## Training-target distribution

The median is preferred to the mean because delay distributions are asymmetric and
contain extreme events. The 15- and 60-minute thresholds are used only for diagnostic
evaluation; the true segment would not be known when making a prediction.


In [3]:
train = splits["train"]
validation = splits["validation"]
test = splits["test"]

target_summary = train.agg(
    F.count("*").alias("rows"),
    F.avg(TARGET).alias("mean_delay"),
    F.stddev(TARGET).alias("std_delay"),
    F.min(TARGET).alias("min_delay"),
    F.max(TARGET).alias("max_delay"),
    F.avg((F.col(TARGET) > 15).cast("double")).alias("delayed_over_15_share"),
    F.avg((F.col(TARGET) > 60).cast("double")).alias("delayed_over_60_share"),
).first().asDict()
quantiles = train.approxQuantile(TARGET, [0.10, 0.50, 0.90, 0.95, 0.99], 0.001)
target_summary["quantiles_p10_p50_p90_p95_p99"] = quantiles
print(target_summary)


{'rows': 2457169, 'mean_delay': 4.381878624004672, 'std_delay': 16.95849905376041, 'min_delay': -119.96666666666667, 'max_delay': 1509.8333333333333, 'delayed_over_15_share': 0.18163056753524076, 'delayed_over_60_share': 0.01048076058260543, 'quantiles_p10_p50_p90_p95_p99': [-12.45, 2.4166666666666665, 22.25, 31.683333333333334, 59.666666666666664]}


## Fit all historical aggregates using train only

Routes with fewer than 100 training flights are not assigned their own estimate.
This reduces unstable medians and gives unseen or sparse routes a deterministic fallback.
Route+airline thresholds are compared later using validation only.


In [4]:
global_median = quantiles[1]
route_medians = (
    train.groupBy("ADEP", "ADES")
    .agg(
        F.count("*").alias("route_train_rows"),
        F.percentile_approx(TARGET, 0.5, 10_000).alias("route_median"),
    )
    .filter(F.col("route_train_rows") >= MIN_ROUTE_OBSERVATIONS)
)
adep_medians = (
    train.groupBy("ADEP")
    .agg(
        F.count("*").alias("adep_train_rows"),
        F.percentile_approx(TARGET, 0.5, 10_000).alias("adep_median"),
    )
    .filter(F.col("adep_train_rows") >= MIN_ROUTE_OBSERVATIONS)
)
route_airline_stats = (
    train.groupBy("ADEP", "ADES", "AC Operator")
    .agg(
        F.count("*").alias("route_airline_train_rows"),
        F.percentile_approx(TARGET, 0.5, 10_000).alias("route_airline_median"),
    )
).persist(StorageLevel.DISK_ONLY)
adep_airline_stats = (
    train.groupBy("ADEP", "AC Operator")
    .agg(
        F.count("*").alias("adep_airline_train_rows"),
        F.percentile_approx(TARGET, 0.5, 10_000).alias("adep_airline_median"),
    )
).persist(StorageLevel.DISK_ONLY)
print({
    "global_train_median": global_median,
    "eligible_routes": route_medians.count(),
    "eligible_departure_airports": adep_medians.count(),
    "train_route_airline_combinations": route_airline_stats.count(),
    "train_departure_airport_airline_combinations": adep_airline_stats.count(),
    "minimum_training_observations": MIN_ROUTE_OBSERVATIONS,
})


{'global_train_median': 2.4166666666666665, 'eligible_routes': 6337, 'eligible_departure_airports': 722, 'train_route_airline_combinations': 42649, 'train_departure_airport_airline_combinations': 9738, 'minimum_training_observations': 100}


In [5]:
def add_baseline_predictions(frame):
    original_rows = frame.count()
    baseline_columns = frame.select("ADEP", "ADES", "AC Operator", TARGET)
    scored = (
        baseline_columns.join(F.broadcast(route_medians), ["ADEP", "ADES"], "left")
        .join(F.broadcast(adep_medians), ["ADEP"], "left")
        .withColumn("prediction_global_median", F.lit(global_median))
        .withColumn(
            "fallback_source",
            F.when(F.col("route_median").isNotNull(), F.lit("route"))
            .when(F.col("adep_median").isNotNull(), F.lit("departure_airport"))
            .otherwise(F.lit("global")),
        )
        .withColumn(
            "prediction_route_fallback",
            F.coalesce("route_median", "adep_median", F.lit(global_median)),
        )
    )
    assert scored.count() == original_rows
    return scored

validation_base_predictions = add_baseline_predictions(validation).persist(
    StorageLevel.DISK_ONLY
)
validation_base_predictions.groupBy("fallback_source").count().orderBy(
    F.desc("count")
).show(truncate=False)

def add_route_airline_prediction(frame, minimum_rows):
    route_airline_eligible = F.col("route_airline_train_rows") >= minimum_rows
    adep_airline_eligible = F.col("adep_airline_train_rows") >= minimum_rows
    return (
        frame.join(
            F.broadcast(route_airline_stats),
            ["ADEP", "ADES", "AC Operator"],
            "left",
        )
        .join(
            F.broadcast(adep_airline_stats),
            ["ADEP", "AC Operator"],
            "left",
        )
        .withColumn(
            "route_airline_fallback_source",
            F.when(route_airline_eligible, F.lit("route_airline"))
            .when(F.col("route_median").isNotNull(), F.lit("route"))
            .when(adep_airline_eligible, F.lit("departure_airport_airline"))
            .when(F.col("adep_median").isNotNull(), F.lit("departure_airport"))
            .otherwise(F.lit("global")),
        )
        .withColumn(
            "prediction_route_airline_fallback",
            F.coalesce(
                F.when(route_airline_eligible, F.col("route_airline_median")),
                F.col("route_median"),
                F.when(adep_airline_eligible, F.col("adep_airline_median")),
                F.col("adep_median"),
                F.lit(global_median),
            ),
        )
    )

validation_combinations = validation.select(
    "ADEP", "ADES", "AC Operator"
).distinct()
unseen_validation_combinations = validation_combinations.join(
    route_airline_stats.select("ADEP", "ADES", "AC Operator"),
    ["ADEP", "ADES", "AC Operator"],
    "left_anti",
).count()

threshold_results = []
fallback_levels = [
    "route_airline", "route", "departure_airport_airline",
    "departure_airport", "global",
]
for threshold in (20, 50, 100, 200):
    candidate = add_route_airline_prediction(
        validation_base_predictions, threshold
    ).persist(StorageLevel.DISK_ONLY)
    errors = candidate.select(
        F.abs(F.col(TARGET) - F.col("prediction_route_airline_fallback")).alias(
            "absolute_error"
        ),
        F.pow(F.col(TARGET) - F.col("prediction_route_airline_fallback"), 2).alias(
            "squared_error"
        ),
    )
    summary = errors.agg(
        F.count("*").alias("rows"),
        F.avg("absolute_error").alias("MAE"),
        F.sqrt(F.avg("squared_error")).alias("RMSE"),
        F.percentile_approx("absolute_error", 0.5, 10_000).alias(
            "median_absolute_error"
        ),
        F.percentile_approx("absolute_error", 0.9, 10_000).alias(
            "p90_absolute_error"
        ),
    ).first().asDict()
    assert summary["rows"] == EXPECTED_COUNTS["validation"]
    coverage = {
        row["route_airline_fallback_source"]: row["count"]
        for row in candidate.groupBy("route_airline_fallback_source").count().collect()
    }
    summary.update({
        "minimum_rows": threshold,
        "unseen_validation_combinations": unseen_validation_combinations,
        "unseen_validation_rows": candidate.filter(
            F.col("route_airline_train_rows").isNull()
        ).count(),
        **{f"{level}_rows": coverage.get(level, 0) for level in fallback_levels},
    })
    threshold_results.append(summary)
    candidate.unpersist()

threshold_results_df = spark.createDataFrame(threshold_results).orderBy("minimum_rows")
threshold_results_df.show(truncate=False)
SELECTED_ROUTE_AIRLINE_THRESHOLD = min(
    threshold_results, key=lambda result: result["MAE"]
)["minimum_rows"]
print({
    "selected_on": "validation MAE",
    "selected_route_airline_threshold": SELECTED_ROUTE_AIRLINE_THRESHOLD,
})
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
THRESHOLD_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
threshold_results_df.toPandas().to_csv(THRESHOLD_REPORT_PATH, index=False)

validation_predictions = add_route_airline_prediction(
    validation_base_predictions, SELECTED_ROUTE_AIRLINE_THRESHOLD
).persist(StorageLevel.DISK_ONLY)
assert validation_predictions.count() == EXPECTED_COUNTS["validation"]
validation_predictions.groupBy("route_airline_fallback_source").count().orderBy(
    F.desc("count")
).show(truncate=False)
validation_base_predictions.unpersist()


+-----------------+------+
|fallback_source  |count |
+-----------------+------+
|route            |470478|
|departure_airport|117849|
|global           |3064  |
+-----------------+------+



+------------------+------------------+------------------------------+----------------------+-----------+---------------------+------------+------------------+------------------+----------+------+------------------------------+----------------------+
|MAE               |RMSE              |departure_airport_airline_rows|departure_airport_rows|global_rows|median_absolute_error|minimum_rows|p90_absolute_error|route_airline_rows|route_rows|rows  |unseen_validation_combinations|unseen_validation_rows|
+------------------+------------------+------------------------------+----------------------+-----------+---------------------+------------+------------------+------------------+----------+------+------------------------------+----------------------+
|9.953347418768805 |15.668838887983782|22918                         |5135                  |957        |7.15                 |20          |20.4              |552210            |10171     |591391|3181                          |15271               

+-----------------------------+------+
|route_airline_fallback_source|count |
+-----------------------------+------+
|route_airline                |552210|
|departure_airport_airline    |22918 |
|route                        |10171 |
|departure_airport            |5135  |
|global                       |957   |
+-----------------------------+------+



DataFrame[ADEP: string, ADES: string, AC Operator: string, Arrival_Delay_Min: double, route_train_rows: bigint, route_median: double, adep_train_rows: bigint, adep_median: double, prediction_global_median: double, fallback_source: string, prediction_route_fallback: double]

## Metrics and diagnostic segments

MAE is the primary selection metric. RMSE exposes large misses, while the median and
P90 absolute errors describe the typical flight and the tail of the error distribution.


In [6]:
def score_baselines(frame, split_name):
    candidates = frame.select(
        TARGET,
        F.explode(F.array(
            F.struct(F.lit("global_median").alias("baseline"),
                     F.col("prediction_global_median").alias("prediction")),
            F.struct(F.lit("route_median_with_fallback").alias("baseline"),
                     F.col("prediction_route_fallback").alias("prediction")),
            F.struct(F.lit("route_airline_median_with_fallback").alias("baseline"),
                     F.col("prediction_route_airline_fallback").alias("prediction")),
        )).alias("candidate"),
    ).select(TARGET, "candidate.*")
    errors = (
        candidates.withColumn("absolute_error",
                              F.abs(F.col(TARGET) - F.col("prediction")))
        .withColumn("squared_error",
                    F.pow(F.col(TARGET) - F.col("prediction"), 2))
    )
    exclusive = errors.withColumn(
        "segment",
        F.when(F.col(TARGET) <= 15, F.lit("punctual_<=15"))
        .when(F.col(TARGET) <= 60, F.lit("moderate_15_60"))
        .otherwise(F.lit("severe_>60")),
    )
    segmented = (
        exclusive.unionByName(errors.withColumn("segment", F.lit("all")))
        .unionByName(errors.filter(F.col(TARGET) > 15).withColumn(
            "segment", F.lit("delayed_>15")
        ))
    )
    return (
        segmented.groupBy("baseline", "segment")
        .agg(
            F.count("*").alias("rows"),
            F.avg("absolute_error").alias("MAE"),
            F.sqrt(F.avg("squared_error")).alias("RMSE"),
            F.percentile_approx("absolute_error", 0.5, 10_000).alias("median_absolute_error"),
            F.percentile_approx("absolute_error", 0.9, 10_000).alias("p90_absolute_error"),
        )
        .withColumn("split", F.lit(split_name))
        .withColumn(
            "route_airline_minimum_rows", F.lit(SELECTED_ROUTE_AIRLINE_THRESHOLD)
        )
        .select("split", "baseline", "segment", "rows", "MAE", "RMSE",
                "median_absolute_error", "p90_absolute_error",
                "route_airline_minimum_rows")
    )

validation_metrics = score_baselines(validation_predictions, "validation").cache()
validation_metrics.orderBy("baseline", "segment").show(30, truncate=False)


+----------+----------------------------------+--------------+------+------------------+------------------+---------------------+------------------+--------------------------+
|split     |baseline                          |segment       |rows  |MAE               |RMSE              |median_absolute_error|p90_absolute_error|route_airline_minimum_rows|
+----------+----------------------------------+--------------+------+------------------+------------------+---------------------+------------------+--------------------------+
|validation|global_median                     |all           |591391|11.685326881876918|17.88509644914023 |8.450000000000001    |24.166666666666668|20                        |
|validation|global_median                     |delayed_>15   |112920|27.34122889361201 |34.80275652284281 |21.166666666666664   |46.25             |20                        |
|validation|global_median                     |moderate_15_60|106402|23.591041051859886|25.64759115737857 |20.4333333333

## Freeze the validation winner, then open test

The winner is selected only by overall validation MAE. Test metrics are computed after
that decision and are not used to change the baseline rules.


In [7]:
best_validation = (
    validation_metrics.filter(F.col("segment") == "all")
    .orderBy(F.asc("MAE"))
    .first()
)
SELECTED_BASELINE = best_validation["baseline"]
print({"selected_on": "validation MAE",
       "selected_baseline": SELECTED_BASELINE,
       "validation_MAE": best_validation["MAE"]})

test_base_predictions = add_baseline_predictions(test).persist(StorageLevel.DISK_ONLY)
test_predictions = add_route_airline_prediction(
    test_base_predictions, SELECTED_ROUTE_AIRLINE_THRESHOLD
).persist(StorageLevel.DISK_ONLY)
assert test_predictions.count() == EXPECTED_COUNTS["test"]
test_predictions.groupBy("route_airline_fallback_source").count().orderBy(
    F.desc("count")
).show(truncate=False)
test_base_predictions.unpersist()
test_metrics = score_baselines(test_predictions, "test")
all_metrics = validation_metrics.unionByName(test_metrics)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
metrics_pd = all_metrics.orderBy("split", "baseline", "segment").toPandas()
display(metrics_pd)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
metrics_pd.to_csv(REPORT_PATH, index=False)
print(f"Metrics saved to {REPORT_PATH}")


{'selected_on': 'validation MAE', 'selected_baseline': 'route_airline_median_with_fallback', 'validation_MAE': 9.953347418768805}


+-----------------------------+------+
|route_airline_fallback_source|count |
+-----------------------------+------+
|route_airline                |575453|
|departure_airport_airline    |26322 |
|route                        |12915 |
|departure_airport            |6354  |
|global                       |1654  |
+-----------------------------+------+



,split,baseline,segment,rows,MAE,RMSE,median_absolute_error,p90_absolute_error,route_airline_minimum_rows
0,test,global_median,all,622698,11.845162,17.907026,8.516667,24.850000,20
1,test,global_median,delayed_>15,121569,27.637317,34.490306,21.750000,47.450000,20
2,test,global_median,moderate_15_60,114386,24.153757,26.276172,20.950000,40.083333,20
3,test,global_median,punctual_<=15,501129,8.014140,10.481897,6.583333,16.633333,20
4,test,global_median,severe_>60,7183,83.111432,95.594046,72.616667,114.583333,20
5,test,route_airline_median_with_fallback,all,622698,10.179615,16.074417,7.133333,21.200000,20
6,test,route_airline_median_with_fallback,delayed_>15,121569,22.326643,30.818301,17.600000,42.850000,20
7,test,route_airline_median_with_fallback,moderate_15_60,114386,18.997375,22.388418,16.800000,35.766667,20
8,test,route_airline_median_with_fallback,punctual_<=15,501129,7.232865,9.521799,5.833333,14.966667,20
9,test,route_airline_median_with_fallback,severe_>60,7183,75.343726,89.957470,68.183333,110.100000,20


Metrics saved to C:\Users\celti\OneDrive - Universidade de Santiago de Compostela\Verano\ML_flights_project\reports\05_baseline_metrics.csv


In [8]:
selected_test_metrics = (
    test_metrics.filter(F.col("baseline") == SELECTED_BASELINE)
    .orderBy(F.when(F.col("segment") == "all", 0).otherwise(1), "segment")
)
selected_test_metrics.show(truncate=False)
print("The four candidate models must beat the selected baseline on validation "
      "and preserve that improvement on the untouched temporal test.")

validation_predictions.unpersist()
validation_metrics.unpersist()
test_predictions.unpersist()
route_airline_stats.unpersist()
adep_airline_stats.unpersist()
spark.stop()


+-----+----------------------------------+--------------+------+------------------+------------------+---------------------+------------------+--------------------------+
|split|baseline                          |segment       |rows  |MAE               |RMSE              |median_absolute_error|p90_absolute_error|route_airline_minimum_rows|
+-----+----------------------------------+--------------+------+------------------+------------------+---------------------+------------------+--------------------------+
|test |route_airline_median_with_fallback|all           |622698|10.179615024725578|16.074417169759812|7.133333333333334    |21.2              |20                        |
|test |route_airline_median_with_fallback|delayed_>15   |121569|22.32664330544784 |30.818301298437024|17.599999999999998   |42.85             |20                        |
|test |route_airline_median_with_fallback|moderate_15_60|114386|18.997374824424877|22.388418367740353|16.799999999999997   |35.766666666666666|20

## Interpretation guardrails

- Segment metrics are diagnostics, not separate deployable models.
- A low error for punctual flights can coexist with poor detection of important delays.
- The post-off-block departure-delay baseline must not be compared directly with this task: it
  has access to later information and answers a different operational question.
- Model selection should prioritise validation MAE while checking RMSE and P90 for tail risk.

### Implemented · Route + airline baseline

`route_airline_median_with_fallback` uses train-only groups of `ADEP`, `ADES` and
`AC Operator`. Thresholds of 20, 50, 100 and 200 flights are compared using validation
MAE, with coverage and unseen combinations reported. It uses this hierarchy:

`route + airline → route → departure airport + airline → departure airport → global`.

The threshold and winning baseline are frozen on validation before consulting test.

### TODO · Optional LightGBM/SynapseML experiment

Use XGBoost as the fourth model in the current Spark 4.2 pipeline. Consider LightGBM
through SynapseML only as a later, separate experiment because the published SynapseML
artifacts target Spark 3.x and Scala 2.12, whereas this project uses Spark 4.2 and Scala
2.13. Such an experiment should use an isolated compatible environment and must not
replace the reproducible XGBoost comparison unless it is validated end to end.
